In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
import plotly.express as px
from astroquery.ned import Ned
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

df = pd.read_csv("batse.csv")

df['RA'] = df['RA']
df['Dec'] = df['Dec']

def query_host_galaxy(ra, dec):
    try:
        result = Ned.query_region(f"{ra} {dec}", radius="0.1 degree")
        if result is not None and len(result) > 0:
            host_galaxy = result['NAME'][0]  
            return host_galaxy
        else:
            return "Unknown"
    except Exception as e:
        return "Error in query"

df['Host_Galaxy'] = df.apply(lambda row: query_host_galaxy(row['RA'], row['Dec']), axis=1)

def classify_grb_binary(t90):
    return "Short" if t90 <= 2 else "Long"

df["Classified"] = df["T90"].apply(classify_grb_binary)

# Progenitor assignment based on classification
def assign_progenitor(grb_class):
    if grb_class == "Short":
        return "Type I (Mergers: NS-NS or NS-BH)"
    elif grb_class == "Long":
        return "Type II (Collapsing Massive Stars)"
    else:
        return "Unclassified"

df["Progenitor_Type"] = df["Classified"].apply(assign_progenitor)

# Encode the target variable
class_mapping = {"Short": 0, "Long": 1}
df["grb_class_label"] = df["Classified"].map(class_mapping)

# Define numerical columns for analysis
numerical_columns = [
    "T90", "T50", "epeak", "alpha", "beta",
    "peak_flux_64ms", "peak_flux_256ms", "peak_flux_1024ms",
    "fluence_channel_1", "fluence_channel_2", "fluence_channel_3", "fluence_channel_4"
]

df = df.dropna(subset=numerical_columns)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[numerical_columns])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, df["grb_class_label"], test_size=0.2, random_state=42
)

# Train the SVM model
clf = SVC(kernel="rbf", random_state=42, probability=True)
clf.fit(X_train, y_train)

# Evaluate the model
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred, average="binary")
precision = precision_score(y_test, y_pred, average="binary")
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")
print(f"Confusion Matrix:\n{conf_matrix}")

# Count the number of Short and Long GRBs before clustering
print("Before Machine Learning Clustering:")
print(df["Classified"].value_counts())

# Perform clustering using DBSCAN
clustering = DBSCAN(eps=2, min_samples=5).fit(X_scaled)
df["Cluster"] = clustering.labels_

# Refine progenitor types based on cluster labels
def refine_progenitor(row):
    if row["Cluster"] == -1:
        return "Unusual Cluster"
    return row["Progenitor_Type"]

df["Refined_Progenitor_Type"] = df.apply(refine_progenitor, axis=1)

# Further refine undefined clusters using KMeans
undefined_data = df[df["Cluster"] == -1][numerical_columns]
kmeans = KMeans(n_clusters=2, random_state=42)
undefined_clusters = kmeans.fit_predict(undefined_data)
df.loc[df["Cluster"] == -1, "Undefined_Cluster"] = undefined_clusters

def refine_progenitor_with_patterns(row):
    if row["Cluster"] == -1:
        return f"New Cluster {int(row['Undefined_Cluster'])}"
    return row["Progenitor_Type"]

df["Refined_Progenitor_Type"] = df.apply(refine_progenitor_with_patterns, axis=1)

# Count the number of Short and Long GRBs after clustering
print("After Machine Learning Clustering:")
print(df["Refined_Progenitor_Type"].value_counts())

# PCA for visualization
pca = PCA(n_components=2)
pca_results = pca.fit_transform(X_scaled)
df["pca_1"] = pca_results[:, 0]
df["pca_2"] = pca_results[:, 1]

# Plot PCA results
fig_pca = px.scatter(
    df,
    x="pca_1",
    y="pca_2",
    color="Refined_Progenitor_Type",
    title="PCA Visualization with Progenitor Types and Clusters",
    labels={"pca_1": "PCA Dimension 1", "pca_2": "PCA Dimension 2"},
    template="plotly"
)
fig_pca.show()

# Plot RA and Dec after clustering
fig_ra_dec = px.scatter(
    df,
    x="RA",
    y="Dec",
    color="Refined_Progenitor_Type",
    title="RA and Dec of GRBs with Clusters",
    labels={"RA": "Right Ascension (RA)", "Dec": "Declination (Dec)"},
    template="plotly"
)
fig_ra_dec.show()

# Analyze new clusters
if "Undefined_Cluster" in df.columns:
    for cluster in sorted(df["Undefined_Cluster"].dropna().unique()):
        print(f"Analysis for New Cluster {int(cluster)}:")
        cluster_data = df[df["Undefined_Cluster"] == cluster]
        print(cluster_data.describe())
else:
    print("No undefined clusters to analyze.")

Accuracy: 0.936046511627907
Recall: 0.9424460431654677
Precision: 0.9776119402985075
Confusion Matrix:
[[ 30   3]
 [  8 131]]
Before Machine Learning Clustering:
Classified
Long     666
Short    193
Name: count, dtype: int64
After Machine Learning Clustering:
Refined_Progenitor_Type
Type II (Collapsing Massive Stars)    612
Type I (Mergers: NS-NS or NS-BH)      189
New Cluster 0                          51
New Cluster 1                           7
Name: count, dtype: int64


Analysis for New Cluster 0:


C:\Users\elife\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\pandas\core\nanops.py:1016: RuntimeWarning:

invalid value encountered in subtract



       Trigger Number         T50  T50 Uncertainty  T50 Start Time  \
count       51.000000   51.000000        51.000000       51.000000   
mean      2481.372549   74.449471         0.816608        7.781510   
std       1204.967916  105.498802         1.548854       57.778924   
min        110.000000    0.012000         0.006000     -196.096000   
25%       1617.000000    7.904000         0.091000        4.096000   
50%       2798.000000   17.024000         0.143000       11.264000   
75%       3298.500000  100.192000         0.489000       22.624000   
max       5478.000000  481.984000         7.923000      164.800000   

              T90  T90 Uncertainty  T90 Start Time        epeak  epeak_error  \
count   51.000000        51.000000       51.000000    51.000000    51.000000   
mean   149.219824         2.376745      -13.128843   403.592745    42.070196   
std    157.676201         4.220969       61.549916   240.211001    65.736840   
min      0.068000         0.006000     -259.58400

In [9]:
import pandas as pd
import numpy as np
import plotly.express as px
from astroquery.ned import Ned

df = pd.read_csv("batse.csv")

df['RA'] = df['RA']
df['Dec'] = df['Dec']

def query_host_galaxy(ra, dec):
    try:
        result = Ned.query_region(f"{ra} {dec}", radius="0.1 degree")
        if result is not None and len(result) > 0:
            host_galaxy = result['NAME'][0] 
            return host_galaxy
        else:
            return "Unknown"
    except Exception as e:
        return "Error in query"

df['Host_Galaxy'] = df.apply(lambda row: query_host_galaxy(row['RA'], row['Dec']), axis=1)

def classify_grb_binary(t90):
    return "Short" if t90 <= 2 else "Long"

df["Classified"] = df["T90"].apply(classify_grb_binary)

def assign_progenitor(grb_class):
    if grb_class == "Short":
        return "Type I (Mergers: NS-NS or NS-BH)"
    elif grb_class == "Long":
        return "Type II (Collapsing Massive Stars)"
    else:
        return "Unclassified"

df["Progenitor_Type"] = df["Classified"].apply(assign_progenitor)

def ra_dec_to_cartesian(ra, dec):
    ra_rad = np.radians(ra)
    dec_rad = np.radians(dec)
    
    x = np.cos(dec_rad) * np.cos(ra_rad)
    y = np.cos(dec_rad) * np.sin(ra_rad)
    z = np.sin(dec_rad)

    return x, y, z

df[['x', 'y', 'z']] = df.apply(lambda row: pd.Series(ra_dec_to_cartesian(row['RA'], row['Dec'])), axis=1)

fig = px.scatter_3d(df, x='x', y='y', z='z', color='Progenitor_Type', 
                    title='3D Spherical Plot of GRBs based on RA and Dec',
                    labels={'x': 'X', 'y': 'Y', 'z': 'Z'},
                    color_continuous_scale='Viridis')

fig.update_layout(scene=dict(
                    xaxis_title='X',
                    yaxis_title='Y',
                    zaxis_title='Z'),
                  title='3D Spherical Plot of GRBs')
fig.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
import plotly.express as px
from astroquery.ned import Ned
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Load the BATSE dataset
df = pd.read_csv("batse.csv")

# RA and Dec columns for host galaxy determination
df['RA'] = df['RA']
df['Dec'] = df['Dec']

# Function to query host galaxy based on RA and Dec
def query_host_galaxy(ra, dec):
    try:
        result = Ned.query_region(f"{ra} {dec}", radius="0.1 degree")
        if result is not None and len(result) > 0:
            host_galaxy = result['NAME'][0]  # Host galaxy name from NED
            return host_galaxy
        else:
            return "Unknown"
    except Exception as e:
        return "Error in query"

df['Host_Galaxy'] = df.apply(lambda row: query_host_galaxy(row['RA'], row['Dec']), axis=1)

# Classify GRBs as Short or Long
def classify_grb_binary(t90):
    return "Short" if t90 <= 2 else "Long"

df["Classified"] = df["T90"].apply(classify_grb_binary)

# Progenitor assignment based on classification
def assign_progenitor(grb_class):
    if grb_class == "Short":
        return "Type I (Mergers: NS-NS or NS-BH)"
    elif grb_class == "Long":
        return "Type II (Collapsing Massive Stars)"
    else:
        return "Unclassified"

df["Progenitor_Type"] = df["Classified"].apply(assign_progenitor)

# Encode the target variable
class_mapping = {"Short": 0, "Long": 1}
df["grb_class_label"] = df["Classified"].map(class_mapping)

# Define numerical columns for analysis
numerical_columns = [
    "T90", "T50", "epeak", "alpha", "beta",
    "peak_flux_64ms", "peak_flux_256ms", "peak_flux_1024ms",
    "fluence_channel_1", "fluence_channel_2", "fluence_channel_3", "fluence_channel_4"
]

# Drop rows with NaN values in numerical columns
df = df.dropna(subset=numerical_columns)

# Standardize the numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[numerical_columns])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, df["grb_class_label"], test_size=0.2, random_state=42
)

# Train the SVM model
clf = SVC(kernel="rbf", random_state=42, probability=True)
clf.fit(X_train, y_train)

# Evaluate the model
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred, average="binary")
precision = precision_score(y_test, y_pred, average="binary")
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")
print(f"Confusion Matrix:\n{conf_matrix}")

# Count the number of Short and Long GRBs before clustering
print("Before Machine Learning Clustering:")
print(df["Classified"].value_counts())

# Perform clustering using DBSCAN
clustering = DBSCAN(eps=2, min_samples=5).fit(X_scaled)
df["Cluster"] = clustering.labels_

# Refine progenitor types based on cluster labels
def refine_progenitor(row):
    if row["Cluster"] == -1:
        return "Unusual Cluster"
    return row["Progenitor_Type"]

df["Refined_Progenitor_Type"] = df.apply(refine_progenitor, axis=1)

undefined_data = df[df["Cluster"] == -1][numerical_columns]
kmeans = KMeans(n_clusters=2, random_state=42)
undefined_clusters = kmeans.fit_predict(undefined_data)
df.loc[df["Cluster"] == -1, "Undefined_Cluster"] = undefined_clusters

def refine_progenitor_with_patterns(row):
    if row["Cluster"] == -1:
        return f"New Cluster {int(row['Undefined_Cluster'])}"
    return row["Progenitor_Type"]

df["Refined_Progenitor_Type"] = df.apply(refine_progenitor_with_patterns, axis=1)

print("After Machine Learning Clustering:")
print(df["Refined_Progenitor_Type"].value_counts())

pca = PCA(n_components=2)
pca_results = pca.fit_transform(X_scaled)
df["pca_1"] = pca_results[:, 0]
df["pca_2"] = pca_results[:, 1]

fig_pca = px.scatter(
    df,
    x="pca_1",
    y="pca_2",
    color="Refined_Progenitor_Type",
    title="PCA Visualization with Progenitor Types and Clusters",
    labels={"pca_1": "PCA Dimension 1", "pca_2": "PCA Dimension 2"},
    template="plotly"
)
fig_pca.show()

fig_ra_dec = px.scatter(
    df,
    x="RA",
    y="Dec",
    color="Refined_Progenitor_Type",
    title="RA and Dec of GRBs with Clusters",
    labels={"RA": "Right Ascension (RA)", "Dec": "Declination (Dec)"},
    template="plotly"
)
fig_ra_dec.show()

if "Undefined_Cluster" in df.columns:
    for cluster in sorted(df["Undefined_Cluster"].dropna().unique()):
        print(f"Analysis for New Cluster {int(cluster)}:")
        cluster_data = df[df["Undefined_Cluster"] == cluster]
        print(cluster_data.describe())
else:
    print("No undefined clusters to analyze.")


Accuracy: 0.936046511627907
Recall: 0.9424460431654677
Precision: 0.9776119402985075
Confusion Matrix:
[[ 30   3]
 [  8 131]]
Before Machine Learning Clustering:
Classified
Long     666
Short    193
Name: count, dtype: int64
After Machine Learning Clustering:
Refined_Progenitor_Type
Type II (Collapsing Massive Stars)    612
Type I (Mergers: NS-NS or NS-BH)      189
New Cluster 0                          51
New Cluster 1                           7
Name: count, dtype: int64


Analysis for New Cluster 0:
       Trigger Number         T50  T50 Uncertainty  T50 Start Time  \
count       51.000000   51.000000        51.000000       51.000000   
mean      2481.372549   74.449471         0.816608        7.781510   
std       1204.967916  105.498802         1.548854       57.778924   
min        110.000000    0.012000         0.006000     -196.096000   
25%       1617.000000    7.904000         0.091000        4.096000   
50%       2798.000000   17.024000         0.143000       11.264000   
75%       3298.500000  100.192000         0.489000       22.624000   
max       5478.000000  481.984000         7.923000      164.800000   

              T90  T90 Uncertainty  T90 Start Time        epeak  epeak_error  \
count   51.000000        51.000000       51.000000    51.000000    51.000000   
mean   149.219824         2.376745      -13.128843   403.592745    42.070196   
std    157.676201         4.220969       61.549916   240.211001    65.736840   
min      0.068000    

C:\Users\elife\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\pandas\core\nanops.py:1016: RuntimeWarning:

invalid value encountered in subtract



In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
from sklearn.cluster import DBSCAN
import plotly.express as px
from sklearn.decomposition import PCA
from astroquery.ned import Ned
from sklearn.cluster import KMeans

df = pd.read_csv("batse.csv")

df['RA'] = df['RA'] 
df['Dec'] = df['Dec'] 

def query_host_galaxy(ra, dec):
    try:
        result = Ned.query_region(f"{ra} {dec}", radius="0.1 degree")
        if result is not None and len(result) > 0:
            host_galaxy = result['NAME'][0]  
            return host_galaxy
        else:
            return "Unknown"
    except Exception as e:
        return "Error in query"

df['Host_Galaxy'] = df.apply(lambda row: query_host_galaxy(row['RA'], row['Dec']), axis=1)


df = pd.read_csv("batse.csv")

if "Undefined_Cluster" in df.columns:
    for cluster in sorted(df["Undefined_Cluster"].dropna().unique()):  
        print(f"Analysis for New Cluster {int(cluster)}:")
        cluster_data = df[df["Undefined_Cluster"] == cluster]
        print(cluster_data.describe())
else:
    print("No undefined clusters to analyze.")

numerical_columns = [
    "T90", "T50", "epeak", "alpha", "beta",
    "peak_flux_64ms", "peak_flux_256ms", "peak_flux_1024ms",
    "fluence_channel_1", "fluence_channel_2", "fluence_channel_3", "fluence_channel_4"
]

def classify_grb_binary(t90):
    return "Short" if t90 <= 2 else "Long"

df["Classified"] = df["T90"].apply(classify_grb_binary)


def assign_progenitor(grb_class):
    if grb_class == "Short":
        return "Type I (Mergers: NS-NS or NS-BH)"
    elif grb_class == "Long":
        return "Type II (Collapsing Massive Stars)"
    else:
        return "Unclassified"

df["Progenitor_Type"] = df["Classified"].apply(assign_progenitor)

class_mapping = {"Short": 0, "Long": 1}
df["grb_class_label"] = df["Classified"].map(class_mapping)

df = df.dropna(subset=numerical_columns)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[numerical_columns])

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, df["grb_class_label"], test_size=0.2, random_state=42
)

clf = SVC(kernel="rbf", random_state=42, probability=True)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred, average="binary")
precision = precision_score(y_test, y_pred, average="binary")
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")
print(f"Confusion Matrix:\n{conf_matrix}")

clustering = DBSCAN(eps=2, min_samples=5).fit(X_scaled)
df["Cluster"] = clustering.labels_

def refine_progenitor(row):
    if row["Cluster"] == -1:  
        return "Unusual Cluster"
    return row["Progenitor_Type"]

df["Refined_Progenitor_Type"] = df.apply(refine_progenitor, axis=1)

pca = PCA(n_components=2)
pca_results = pca.fit_transform(X_scaled)

df["pca_1"] = pca_results[:, 0]
df["pca_2"] = pca_results[:, 1]

fig_pca = px.scatter(
    df,
    x="pca_1",
    y="pca_2",
    color="Refined_Progenitor_Type",
    title="PCA Visualization with Progenitor Types",
    labels={"pca_1": "PCA Dimension 1", "pca_2": "PCA Dimension 2"},
    template="plotly"
)
fig_pca.show()

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

undefined_data = df[df["Cluster"] == -1][numerical_columns]

kmeans = KMeans(n_clusters=3, random_state=42)  
undefined_clusters = kmeans.fit_predict(undefined_data)

df.loc[df["Cluster"] == -1, "Undefined_Cluster"] = undefined_clusters

def refine_progenitor_with_patterns(row):
    if row["Cluster"] == -1:
        return f"New Cluster {int(row['Undefined_Cluster'])}"
    return row["Progenitor_Type"]

df["Refined_Progenitor_Type"] = df.apply(refine_progenitor_with_patterns, axis=1)

pca = PCA(n_components=2)
pca_results = pca.fit_transform(X_scaled)

df["pca_1"] = pca_results[:, 0]
df["pca_2"] = pca_results[:, 1]

color_map = {
    "Type I (Mergers: NS-NS or NS-BH)": "blue",
    "Type II (Collapsing Massive Stars)": "green",
    "Unusual Cluster": "red",
    "New Cluster 0": "orange",
    "New Cluster 1": "purple",
}

fig_pca = px.scatter(
    df,
    x="pca_1",
    y="pca_2",
    color="Refined_Progenitor_Type",
    title="PCA Visualization with Refined Progenitor Types and Clusters",
    labels={"pca_1": "PCA Dimension 1", "pca_2": "PCA Dimension 2"},
    color_discrete_map=color_map,
    template="plotly"
)
fig_pca.show()

if "Undefined_Cluster" in df.columns:
    for cluster in sorted(df["Undefined_Cluster"].dropna().unique()): 
        print(f"Analysis for New Cluster {int(cluster)}:")
        cluster_data = df[df["Undefined_Cluster"] == cluster]
        print(cluster_data.describe())
else:
    print("No undefined clusters to analyze.")



No undefined clusters to analyze.
Accuracy: 0.936046511627907
Recall: 0.9424460431654677
Precision: 0.9776119402985075
Confusion Matrix:
[[ 30   3]
 [  8 131]]


Analysis for New Cluster 0:
       Trigger Number         T50  T50 Uncertainty  T50 Start Time  \
count       50.000000   50.000000        50.000000       50.000000   
mean      2450.220000   75.487900         0.827660        7.860340   
std       1196.275313  106.306268         1.562546       58.362757   
min        110.000000    0.012000         0.006000     -196.096000   
25%       1613.000000    7.728000         0.091000        4.576000   
50%       2665.500000   16.768000         0.143000       11.424000   
75%       3244.000000  100.208000         0.502500       23.472000   
max       5478.000000  481.984000         7.923000      164.800000   

              T90  T90 Uncertainty  T90 Start Time       epeak  epeak_error  \
count   50.000000        50.000000       50.000000   50.000000    50.000000   
mean   151.027900         2.324400      -13.397820  388.984600    38.211600   
std    158.742052         4.247068       62.144519  218.571503    60.288466   
min      0.068000        

C:\Users\elife\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\pandas\core\nanops.py:1016: RuntimeWarning:

invalid value encountered in subtract

